# Compte rendu individuel — Science des données appliquée au commerce international
**ENCG Settat — S7 Commerce international — BD & DS** · Enseignant : A. Larhlimi · Année universitaire 2026–2027

| | |
|---|---|
| Nom et prénom | **[À compléter]** |
| Code Apogée, groupe | **[À compléter]** |
| Sujet attribué | CI**[XX]** — **[titre]** |
| Date | **[JJ/MM/AAAA]** |
| Dépôt GitHub | **[URL]** |

> ⚠️ Les passages **[À compléter]** sont personnels (identité, sujet attribué, algorithme choisi, déclaration IA). Le reste a été préparé à partir du fichier UN Comtrade fourni : relancez tout le notebook (*Kernel → Restart & Run All*) avant l'export PDF.

## 1. Sujet et problématique

**Contexte commercial.** Le sucre (SH 17) et les céréales (SH 10) sont des produits de base échangés en volumes massifs. Leur prix à l'importation varie fortement selon le produit exact (sucre brut, raffiné, riz, blé, maïs…), le pays importateur, le pays fournisseur et le mode de transport. Pour un importateur ou un négociant, savoir si le prix payé est « normal » pour un flux donné aide à négocier, détecter des anomalies de facturation et budgéter les achats. Les données douanières (UN Comtrade, niveau ligne tarifaire) permettent de calculer une **valeur unitaire** (USD par kg) pour chaque flux bilatéral.

**Problématique.** Peut-on prédire la valeur unitaire (USD/kg) d'un flux d'importation de sucre ou de céréales en 2025 à partir du pays importateur, du pays partenaire, du produit SH6, du mode de transport, du régime douanier et du volume ?

**Type de tâche.** Régression supervisée (cible : logarithme de la valeur unitaire).

**Objectif mesurable (fixé avant le test).** Obtenir un R² ≥ 0,50 sur le test et réduire l'erreur absolue moyenne (MAE, en log) d'au moins 30 % par rapport à la référence naïve « médiane par code SH6 ».

In [ ]:
import warnings, platform, os, glob
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

SEED = 42
np.random.seed(SEED)
os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

# Chemin du fichier : dossier data/ du dépôt, sinon dossier courant
candidats = glob.glob("data/UNComtrade-tariffline*.csv") + glob.glob("UNComtrade-tariffline*.csv") \
          + glob.glob("/mnt/user-data/uploads/UNComtrade-tariffline*.csv")
CSV = candidats[0]
print("Fichier utilisé :", CSV)

## 2. Jeu de données

- **Organisme / source** : UN Comtrade (Division de statistique de l'ONU), base *Tariffline*, données officielles déclarées par les pays.
- **URL** : https://comtradeplus.un.org — **[À compléter : URL exacte de la requête]**
- **Date d'extraction** : 19/11/2026 (d'après le nom du fichier) · **Licence** : conditions d'utilisation UN Comtrade **[À vérifier et citer]**.
- **Période / fréquence** : 2025, annuelle.
- **Flux** : importations uniquement (`flowCode = M`), valeur `primaryValue` en USD, poids net en kg.
- **Codes SH** : chapitres 10 (céréales) et 17 (sucres et sucreries), lignes tarifaires nationales de 6 à 12 chiffres.
- **Limite d'extraction** : le fichier contient exactement 100 000 lignes, ce qui correspond à la **limite d'export** de l'interface. L'échantillon peut donc être **tronqué** et non représentatif de tous les pays (voir section Analyse critique).

In [ ]:
raw = pd.read_csv(CSV, encoding="utf-8-sig", low_memory=False)
print("Dimensions brutes :", raw.shape)
print("Période :", raw.period.unique(), "| flux :", raw.flowDesc.unique())
print("Importateurs :", raw.reporterISO.nunique(), "| partenaires :", raw.partnerISO.nunique(),
      "| codes tarifaires :", raw.cmdCode.nunique())

dico = pd.DataFrame({
    "Variable": ["reporterISO", "partnerISO", "cmdCode", "motDesc", "customsDesc", "netWgt", "primaryValue", "qtyUnitAbbr"],
    "Description et unité": ["Pays importateur (ISO3)", "Pays fournisseur (ISO3)", "Code tarifaire national (SH 6 à 12 chiffres)",
                             "Mode de transport", "Régime douanier", "Poids net (kg)", "Valeur du flux (USD)", "Unité de quantité"],
    "Rôle": ["Explicative", "Explicative", "Explicative (via SH6, SH4)", "Explicative", "Explicative", "Explicative (log) + calcul de la cible",
             "Calcul de la cible", "Filtre (kg uniquement)"],
})
dico["Manquants (%)"] = [round(raw[v].isna().mean() * 100, 1) for v in dico.Variable]
dico

**Adéquation et limites.** Les données couvrent 100+ importateurs et 230+ partenaires, ce qui offre de la diversité. Limites : (1) une seule année, donc aucune dimension temporelle ; (2) la valeur unitaire est une moyenne par ligne douanière, sensible aux erreurs de déclaration de poids ; (3) `cmdDesc` est vide, il faut interpréter les codes SH soi-même ; (4) la troncature à 100 000 lignes peut sur-représenter certains pays.

## 3. Exploration

In [ ]:
print("Valeurs manquantes (%) — principales colonnes")
print((raw[["qty", "netWgt", "cifvalue", "fobvalue", "primaryValue", "customsDesc", "partner2Desc"]].isna().mean() * 100).round(1).to_string())
print("\nLongueur des codes tarifaires :", raw.cmdCode.astype(str).str.len().value_counts().sort_index().to_dict())
print("Unités de quantité :", raw.qtyUnitAbbr.value_counts(dropna=False).to_dict())
print("Valeurs <= 0 : primaryValue", (raw.primaryValue <= 0).sum(), "| netWgt", (raw.netWgt <= 0).sum())
raw[["netWgt", "primaryValue"]].describe().round(0)

In [ ]:
# Construction de la base d'analyse (voir section 4 pour la justification)
df = raw.copy()
df["cmd"] = df.cmdCode.astype(str)
df = df[(df.cmd.str.len() >= 6) & (df.qtyUnitAbbr == "kg") & (df.netWgt > 0) & (df.primaryValue > 0)].copy()
df["hs2"], df["hs4"], df["hs6"] = df.cmd.str[:2], df.cmd.str[:4], df.cmd.str[:6]
df["uv"] = df.primaryValue / df.netWgt                      # valeur unitaire USD/kg
lo, hi = df.uv.quantile([.005, .995])
df = df[(df.uv > lo) & (df.uv < hi)].copy()                  # retrait des 1 % de valeurs extrêmes
df["y"] = np.log(df.uv)                                      # cible
df["lw"] = np.log(df.netWgt)                                 # log du poids
df["mot"] = df.motDesc.fillna("NA"); df["cust"] = df.customsDesc.fillna("NA")
df["grp"] = df.reporterISO + "_" + df.partnerISO + "_" + df.hs6   # clé de groupe anti-fuite
print(f"Lignes conservées : {len(df):,} sur {len(raw):,} ({len(df)/len(raw):.1%})")
print(f"Bornes de valeur unitaire retenues : {lo:.3f} à {hi:.1f} USD/kg")

In [ ]:
# Figure 1 — distribution de la valeur unitaire par chapitre
fig, ax = plt.subplots(figsize=(7, 4))
for ch, lab in [("10", "SH 10 — Céréales"), ("17", "SH 17 — Sucres")]:
    ax.hist(df.loc[df.hs2 == ch, "y"], bins=60, alpha=.6, label=lab, density=True)
ax.set_xlabel("log(valeur unitaire) — log(USD par kg)"); ax.set_ylabel("Densité")
ax.set_title("Figure 1 — Valeur unitaire à l'importation, 2025"); ax.legend()
plt.tight_layout(); plt.savefig("figures/fig1_distribution.png"); plt.show()

**Interprétation 1.** Les deux chapitres ont des distributions très différentes : les céréales se concentrent autour de prix bas (quelques dixièmes d'USD/kg), tandis que les sucres et sucreries s'étalent sur une plage plus large avec une queue vers les produits transformés à forte valeur. Le chapitre SH seul explique déjà une partie de la variance, mais pas tout.

In [ ]:
# Figure 2 — principaux importateurs par valeur
top = df.groupby("reporterISO").primaryValue.sum().sort_values(ascending=False).head(10) / 1e9
fig, ax = plt.subplots(figsize=(7, 4))
top[::-1].plot.barh(ax=ax, color="#1D4ED8")
ax.set_xlabel("Valeur importée en 2025 (milliards USD)"); ax.set_ylabel("Pays importateur")
ax.set_title("Figure 2 — Dix premiers importateurs dans l'échantillon"); ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.savefig("figures/fig2_importateurs.png"); plt.show()

**Interprétation 2.** Quelques importateurs concentrent l'essentiel de la valeur échantillonnée. Cette concentration est en partie réelle (grands marchés) et en partie un effet de la troncature à 100 000 lignes : à lire avec prudence, elle ne représente pas le commerce mondial complet.

In [ ]:
# Figure 3 — valeur unitaire par position SH4 (les 8 plus fréquentes)
top4 = df.hs4.value_counts().head(8).index
fig, ax = plt.subplots(figsize=(8, 4))
data = [df.loc[df.hs4 == h, "y"] for h in top4]
ax.boxplot(data, showfliers=False)
ax.set_xticks(range(1, len(top4) + 1)); ax.set_xticklabels(list(top4))
ax.set_xlabel("Position SH4"); ax.set_ylabel("log(valeur unitaire) — log(USD par kg)")
ax.set_title("Figure 3 — Dispersion des prix par position SH4")
plt.tight_layout(); plt.savefig("figures/fig3_sh4.png"); plt.show()

**Interprétation 3.** Le produit (position SH4) structure fortement le prix, mais la dispersion à l'intérieur d'une même position reste large : c'est là que le pays importateur, le fournisseur et le transport peuvent apporter de l'information supplémentaire. Cela justifie un modèle non linéaire capable de combiner des variables catégorielles.

## 4. Préparation et prévention des fuites

**Nettoyage et nouvelles variables (décisions justifiées).**
- On garde les lignes dont le code tarifaire a au moins 6 chiffres (les codes plus courts sont des agrégats ambigus).
- On garde uniquement l'unité *kg*, pour que la valeur unitaire soit comparable.
- On retire les poids et valeurs ≤ 0 (division impossible), puis les 0,5 % de valeurs unitaires les plus basses et les plus hautes (erreurs probables de déclaration).
- Nouvelles variables : `hs4`, `hs6` (niveaux de la nomenclature), `lw` = log du poids net, cible `y` = log(valeur unitaire). Le logarithme rend la cible plus symétrique et traduit des écarts en pourcentage.

**Découpage.** Le même trio importateur–fournisseur–produit apparaît souvent sur plusieurs lignes (régimes douaniers, transports différents). Un découpage aléatoire simple mettrait des lignes quasi identiques dans le train et le test : c'est une **fuite**. On découpe donc **par groupe** (`reporter_partner_hs6`) : 80 % des groupes en entraînement, 20 % en test. Il n'y a pas de date exploitable (une seule année), donc pas de découpage temporel.

**Prétraitements entraînés uniquement sur le train.** L'encodage des catégories est placé **dans le `Pipeline`** : il est ajusté sur le train seulement. La référence naïve est aussi calculée sur le train seulement.

In [ ]:
cat = ["reporterISO", "partnerISO", "hs6", "hs4", "mot", "cust"]
num = ["lw"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
i_tr, i_te = next(gss.split(df, groups=df.grp))
train, test = df.iloc[i_tr].copy(), df.iloc[i_te].copy()

# Preuve d'absence de fuite : aucun groupe commun entre train et test
assert set(train.grp).isdisjoint(set(test.grp))
print(f"Train : {len(train):,} lignes, {train.grp.nunique():,} groupes")
print(f"Test  : {len(test):,} lignes, {test.grp.nunique():,} groupes")
print("Groupes communs :", len(set(train.grp) & set(test.grp)))

## 5. Algorithme principal

**Nom et numéro du support :** Forêt aléatoire de régression (Random Forest), numéro **[À compléter : n° parmi les 29]**.
> ⚠️ Vérifiez que cet algorithme figure bien dans la liste imposée par l'énoncé et correspond à votre sujet. Sinon, remplacez-le dans la cellule du modèle (section 6) ; le reste du notebook ne change pas.

**Pourquoi cet algorithme et non une alternative ?** Les variables explicatives sont surtout catégorielles à nombreuses modalités (pays, codes SH) avec des interactions : le prix d'un produit dépend du couple importateur–fournisseur. Une régression linéaire simple suppose des effets additifs et exigerait un encodage très large ; un arbre unique est instable et sur-apprend. La forêt moyenne beaucoup d'arbres décorrélés, ce qui réduit la variance, capture les interactions sans les spécifier, et reste robuste aux échelles et aux valeurs extrêmes. Elle fournit aussi des mesures d'importance des variables, utiles pour l'interprétation commerciale. Son inconvénient est un modèle peu transparent et plus lourd en mémoire.

**Formule essentielle.**

$$\hat{y}(x) = \frac{1}{B}\sum_{b=1}^{B} T_b(x)$$

où $\hat{y}(x)$ est la prédiction du log de la valeur unitaire pour la ligne $x$ ; $B$ le nombre d'arbres ; $T_b$ le $b$-ième arbre, entraîné sur un échantillon bootstrap des données et, à chaque coupure, sur un sous-ensemble aléatoire de variables ; $x$ le vecteur des variables explicatives (pays, produit, transport, régime, log du poids). Chaque coupure d'un arbre minimise la variance interne des deux nœuds fils : $\sum_{i\in G}(y_i-\bar y_G)^2$, avec $G$ l'ensemble des observations d'un nœud, $y_i$ la cible de l'observation $i$ et $\bar y_G$ la moyenne de la cible dans $G$.

**Paramètres appris** : les variables et seuils de coupure de chaque arbre et les valeurs moyennes dans les feuilles. **Deux hyperparamètres réglés** : `min_samples_leaf` (taille minimale d'une feuille, contrôle du surapprentissage) et `max_features` (part des variables testées à chaque coupure, contrôle de la corrélation entre arbres).

**Schéma de fonctionnement :** *[À compléter : insérer un schéma original dessiné par vous — échantillons bootstrap → arbres indépendants → moyenne des prédictions]*.

## 6. Protocole expérimental

- **Référence naïve** : médiane de `y` par code SH6 calculée sur le train (repli sur SH4, puis sur la médiane globale pour un code absent du train).
- **Validation** : validation croisée par groupes (`GroupKFold`, 3 plis) **sur le train uniquement**. Le test n'est utilisé qu'**une seule fois**, à la fin.
- **Métriques** : principale = R² (part de variance expliquée) ; secondaires = MAE et RMSE en log, parlants en écart relatif de prix (une MAE de 0,5 en log ≈ ±65 %).
- **Réglage** : 6 configurations (`min_samples_leaf` ∈ {1, 3, 10} × `max_features` ∈ {0,5 ; 1,0}).

In [ ]:
def prediction_naive(tr, te):
    m6 = tr.groupby("hs6").y.median()
    m4 = tr.groupby("hs4").y.median()
    return te.hs6.map(m6).fillna(te.hs4.map(m4)).fillna(tr.y.median()).values

def pipeline(leaf, mf, n_trees=100):
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat),
        ("num", "passthrough", num)])
    rf = RandomForestRegressor(n_estimators=n_trees, min_samples_leaf=leaf, max_features=mf,
                               n_jobs=-1, random_state=SEED)
    return Pipeline([("prep", pre), ("rf", rf)])

# Réglage par validation croisée groupée, sur le train uniquement
gkf = GroupKFold(n_splits=3)
essais = []
for leaf in [1, 3, 10]:
    for mf in [0.5, 1.0]:
        scores, maes = [], []
        for a, b in gkf.split(train, groups=train.grp):
            fa, fb = train.iloc[a], train.iloc[b]
            m = pipeline(leaf, mf).fit(fa[cat + num], fa.y)
            p = m.predict(fb[cat + num])
            scores.append(r2_score(fb.y, p)); maes.append(mean_absolute_error(fb.y, p))
        essais.append({"min_samples_leaf": leaf, "max_features": mf, "R2 validation": np.mean(scores), "MAE validation": np.mean(maes)})
essais = pd.DataFrame(essais).sort_values("R2 validation", ascending=False).reset_index(drop=True)
essais.round(3)

## 7. Résultats et interprétation

In [ ]:
best = essais.iloc[0]
print("Meilleure configuration :", best[["min_samples_leaf", "max_features"]].to_dict())

# Référence naïve : validation (CV groupée) et test
val_naif = []
for a, b in gkf.split(train, groups=train.grp):
    fa, fb = train.iloc[a], train.iloc[b]
    val_naif.append((r2_score(fb.y, prediction_naive(fa, fb)), mean_absolute_error(fb.y, prediction_naive(fa, fb))))
val_naif = np.mean(val_naif, axis=0)
p_naif = prediction_naive(train, test)

# Modèle final : entraîné sur tout le train, évalué UNE fois sur le test
final = pipeline(int(best.min_samples_leaf), float(best.max_features), n_trees=300).fit(train[cat + num], train.y)
p_rf = final.predict(test[cat + num])

res = pd.DataFrame({
    "Méthode": ["Référence (médiane SH6)", "Forêt aléatoire"],
    "R² validation": [val_naif[0], best["R2 validation"]],
    "R² test": [r2_score(test.y, p_naif), r2_score(test.y, p_rf)],
    "MAE test (log)": [mean_absolute_error(test.y, p_naif), mean_absolute_error(test.y, p_rf)],
    "RMSE test (log)": [root_mean_squared_error(test.y, p_naif), root_mean_squared_error(test.y, p_rf)],
}).round(3)
gain = 1 - res.loc[1, "MAE test (log)"] / res.loc[0, "MAE test (log)"]
print(f"Réduction de la MAE par rapport à la référence : {gain:.1%}")
print("Objectif R² ≥ 0,50 :", "ATTEINT" if res.loc[1, "R² test"] >= .5 else "NON atteint",
      "| objectif −30 % de MAE :", "ATTEINT" if gain >= .30 else "NON atteint")
res

In [ ]:
# Figure 4 — valeurs prédites vs observées (test)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(test.y, p_rf, s=4, alpha=.25)
lim = [min(test.y.min(), p_rf.min()), max(test.y.max(), p_rf.max())]
ax.plot(lim, lim, "r--", lw=1, label="prédiction parfaite")
ax.set_xlabel("log(valeur unitaire) observé"); ax.set_ylabel("log(valeur unitaire) prédit")
ax.set_title("Figure 4 — Prédit vs observé (jeu de test)"); ax.legend()
plt.tight_layout(); plt.savefig("figures/fig4_pred_vs_obs.png"); plt.show()

# Erreurs par chapitre et par grand importateur
test["err"] = (test.y - p_rf).abs()
print("MAE par chapitre SH :\n", test.groupby("hs2").err.mean().round(3).to_string())
gros = test.reporterISO.value_counts().head(6).index
print("\nMAE par importateur (6 plus fréquents) :\n", test[test.reporterISO.isin(gros)].groupby("reporterISO").err.mean().round(3).to_string())

In [ ]:
# Figure 5 — importance des variables par permutation (échantillon du test)
ech = test.sample(min(6000, len(test)), random_state=SEED)
pi = permutation_importance(final, ech[cat + num], ech.y, n_repeats=3, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=cat + num).sort_values()
fig, ax = plt.subplots(figsize=(6.5, 3.8))
imp.plot.barh(ax=ax, color="#0F766E")
ax.set_xlabel("Baisse de R² quand la variable est mélangée"); ax.set_title("Figure 5 — Importance des variables")
ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.savefig("figures/fig5_importance.png"); plt.show()
imp.sort_values(ascending=False).round(3)

**Interprétation.** *(À relire après exécution, avec vos chiffres réels.)* Comparez le R² et la MAE du modèle à ceux de la référence : l'écart mesure l'information apportée par le pays importateur, le fournisseur, le transport et le volume au-delà du simple code SH6. Le graphique « prédit vs observé » montre où le modèle sous-estime les prix extrêmes ; l'importance par permutation indique quelles variables pèsent le plus.

**Analyse des erreurs, surapprentissage et robustesse.** *[À compléter avec vos chiffres :]* comparez le R² de validation et le R² de test (un grand écart signalerait un surapprentissage) ; commentez les chapitres et importateurs pour lesquels la MAE est la plus élevée ; indiquez la sensibilité au choix de la graine ou du découpage si vous l'avez testée.

**Exemple d'usage commercial.** Un acheteur d'un importateur reçoit une offre de sucre raffiné (SH 1701) d'un fournisseur donné. Il compare le prix proposé à la valeur unitaire prédite pour ce couple pays–fournisseur–produit : un prix très au-dessus de la prédiction (hors de l'erreur typique) justifie une renégociation ou une vérification de la facture.

## 8. Analyse critique

**Biais et limites.**
1. *Échantillon tronqué :* l'export plafonné à 100 000 lignes peut sur-représenter certains pays et en omettre d'autres ; les conclusions ne valent que pour cet échantillon.
2. *Une seule année et valeur unitaire moyenne :* pas d'effet temporel (cours mondiaux, saisonnalité), et les erreurs de déclaration de poids se répercutent directement sur la cible. De plus, le modèle ne voit ni la qualité du produit, ni les termes de livraison (Incoterms), ni les droits de douane.

**Améliorations réalisables.**
1. Télécharger plusieurs années et par importateur pour obtenir un jeu complet, puis tester un découpage temporel (entraîner sur 2022–2024, tester sur 2025).
2. Ajouter des variables externes (distance entre pays, indice des prix mondiaux du sucre et des céréales) et comparer avec un modèle de gradient boosting.

## 9. Conclusion

*[À finaliser avec vos résultats chiffrés :]* répondre explicitement à la problématique (objectif atteint ou non, avec R² et MAE du test comparés à la référence) et en tirer l'implication pour le commerce international : les prix à l'importation de produits de base sont en partie prévisibles à partir du couple pays–fournisseur et du produit, ce qui offre un repère de négociation et de contrôle, dans les limites d'un échantillon d'une seule année.

## 10. Références et reproductibilité

In [ ]:
print("Python", platform.python_version())
for lib in [np, pd, sklearn, matplotlib]:
    print(lib.__name__, lib.__version__)
print("Graine aléatoire :", SEED)

- **Source des données :** UN Comtrade, base Tariffline — **[À compléter : URL exacte, date de consultation 19/11/2026]**.
- **Bibliothèques :** pandas, NumPy, scikit-learn, Matplotlib (versions ci-dessus). Reportez-les dans `requirements.txt`.
- **Dépôt GitHub :** **[À compléter]**.

## Déclaration des outils d'IA
**[À compléter]** : indiquer l'outil utilisé (par exemple Claude), l'usage précis (par exemple aide à la structuration du notebook et au code d'exploration), les cellules concernées, et les vérifications et tests faits personnellement (relecture, exécution complète, contrôle des résultats). Cette déclaration doit rester honnête et à votre nom.